In [1]:
from eeg_bci.data.paths import resolve_data_dir

data_dir = resolve_data_dir("/workspaces/BRAINDECODE/data/moabb")
data_dir

PosixPath('/workspaces/BRAINDECODE/data/moabb')

In [2]:
from braindecode.datasets import MOABBDataset
import moabb

moabb.set_download_dir(str(data_dir))


subject_id = 3

dataset = MOABBDataset(dataset_name="Liu2024", subject_ids=[subject_id])

Used Annotations descriptions: [np.str_('left_hand'), np.str_('right_hand')]


/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:521: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)


In [2]:
from braindecode.datasets import MOABBDataset
import moabb

moabb.set_download_dir(str(data_dir))

subject_id = 3
dataset = MOABBDataset(dataset_name="BNCI2014_001", subject_ids=[subject_id])

In [3]:
import numpy as np

from braindecode.preprocessing import (
    Preprocessor,
    exponential_moving_standardize,
    preprocess,
)

# Motor imagery band
low_cut_hz = 8.0
high_cut_hz = 30.0

# Exponential moving standardization
factor_new = 1e-3
init_block_size = 1000

preprocessors = [
    # Keep only EEG channels
    Preprocessor("pick_types", eeg=True, meg=False, stim=False, eog=False),

    # Convert from V to µV
    Preprocessor(
        lambda data, factor: np.multiply(data, factor),
        factor=1e6,
    ),

    # Band-pass filter for MI rhythms
    Preprocessor(
        "filter",
        l_freq=low_cut_hz,
        h_freq=high_cut_hz,
    ),

    # Standardize each continuous recording
    Preprocessor(
        exponential_moving_standardize,
        factor_new=factor_new,
        init_block_size=init_block_size,
    ),
]

preprocess(dataset, preprocessors, n_jobs=-1)

/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:78: UserWarning: apply_on_array can only be True if fn is a callable function. Automatically correcting to apply_on_array=False.
  warn(
/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:76: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 825 samples (1.650 s)



<BaseConcatDataset | 1 RawDataset(s) | 160000 total samples>
  Sfreq*: 500.0 Hz
  Channels*: 29 (29 EEG)
  Ch. names*: FP1, FP2, Fz, F3, F4, F7, F8, FCz, FC3, FC4, ... (+19 more)
  Montage*: head
  Duration*: 320.0 s
  (* from first recording)
  Description: 1 recordings × 3 columns [subject, session, run]

In [ ]:
import numpy as np

from braindecode.preprocessing import (
    Preprocessor,
    exponential_moving_standardize,
    preprocess,
)

low_cut_hz = 4.0  # low cut frequency for filtering
high_cut_hz = 38.0  # high cut frequency for filtering
# Parameters for exponential moving standardization
factor_new = 1e-3
init_block_size = 1000

preprocessors = [
    Preprocessor("pick_types", eeg=True, meg=False, stim=False),  # Keep EEG sensors
    Preprocessor(
        lambda data, factor: np.multiply(data, factor),  # Convert from V to uV
        factor=1e6,
    ),
    Preprocessor("filter", l_freq=low_cut_hz, h_freq=high_cut_hz),  # Bandpass filter
    Preprocessor(
        exponential_moving_standardize,  # Exponential moving standardization
        factor_new=factor_new,
        init_block_size=init_block_size,
    ),
]

# Preprocess the data
preprocess(dataset, preprocessors, n_jobs=-1)

/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:78: UserWarning: apply_on_array can only be True if fn is a callable function. Automatically correcting to apply_on_array=False.
  warn(
/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:76: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 38 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.00 Hz)
- Upper passband edge: 38.00 Hz
- Upper transition bandwidth: 9.50 Hz (-6 dB cutoff frequency: 42.75 Hz)
- Filter length: 413 samples (1.652 s)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 38 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple an

<BaseConcatDataset | 12 RawDataset(s) | 1160820 total samples>
  Sfreq*: 250.0 Hz
  Channels*: 22 (22 EEG)
  Ch. names*: Fz, FC3, FC1, FCz, FC2, FC4, C5, C3, C1, Cz, ... (+12 more)
  Montage*: head
  Duration*: 386.9 s
  (* from first recording)
  Description: 12 recordings × 3 columns [subject, session, run]

In [4]:
from braindecode.preprocessing import create_windows_from_events

trial_start_offset_seconds = -0.5
# Extract sampling frequency, check that they are same in all datasets
sfreq = dataset.datasets[0].raw.info["sfreq"]
assert all([ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets])
# Calculate the window start offset in samples.
trial_start_offset_samples = int(trial_start_offset_seconds * sfreq)

# Create windows using braindecode function for this. It needs parameters to
# define how windows should be used.
windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=trial_start_offset_samples,
    trial_stop_offset_samples=0,
    preload=True,
)

In [4]:
from braindecode.preprocessing import create_windows_from_events

# Liu2024 sampling rate should be 500 Hz
sfreq = dataset.datasets[0].raw.info["sfreq"]
assert all(ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets)

mi_window_seconds = 4.0
window_size_samples = int(mi_window_seconds * sfreq)

windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=0,
    trial_stop_offset_samples=0,
    window_size_samples=window_size_samples,
    window_stride_samples=window_size_samples,
    mapping={
        "left_hand": 0,
        "right_hand": 1,
    },
    picks="eeg",
    preload=True,
)

/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/braindecode/preprocessing/windowers.py:184: UserWarning: Using reject or picks or flat or dropping bad windows means mne Epochs are created, which will be substantially slower and may be deprecated in the future.
  warnings.warn(


In [7]:
print(windows_dataset.description)
print(len(windows_dataset))

metadata = windows_dataset.get_metadata()
print(metadata.head())
print(metadata.shape)
print(metadata["target"].value_counts())

   subject session run
0        3       0   0
40
   i_window_in_trial  i_start_in_trial  i_stop_in_trial  target  subject  \
0                  0              1008             3008       0        3   
1                  0              5003             7003       1        3   
2                  0              9003            11003       0        3   
3                  0             13003            15003       1        3   
4                  0             17003            19003       0        3   

  session run  
0       0   0  
1       0   0  
2       0   0  
3       0   0  
4       0   0  
(40, 7)
target
0    20
1    20
Name: count, dtype: int64


In [10]:
splitted = windows_dataset.split("session")
train_set = splitted["0train"]  # Session train
test_set = splitted["1test"]  # Session evaluation